# ShapeUQ: Propagating 3D Reconstruction Uncertainty Through Scientific PDE Simulations via Shape Calculus

Reimplementation of the CVPR 2026 3D4S workshop paper (oral). The code was lost, so this notebook rebuilds the whole pipeline from the paper's specification:

1. Neural SDF reconstruction from a noisy point cloud (8-layer MLP, 10-frequency positional encoding, Eikonal regularisation).
2. A Matérn-3/2 Gaussian process fitted by maximum marginal likelihood to the SDF residuals on held-out measurements (Eq. 1).
3. Meshfree collocation PDE solvers (a 4-layer MLP) for the forward problem and the adjoint problem, with domain points sampled where phi < 0 and boundary points found by zero-crossing along the SDF gradient (Section 5, Steps 1 to 3).
4. The Hadamard sensitivity field of Theorem 3 assembled at boundary points, and the GP-propagated variance of Eq. 8, giving a confidence interval from one forward and one adjoint solve.
5. The SciUQ-3D benchmark: glacier heat diffusion, protein electrostatics (linearised Poisson-Boltzmann on a van der Waals surface built from real PDB atoms), and viscous Stokes flow over a coral reef, at 1%, 5% and 10% noise, with the Deterministic, first-order Perturbation and Monte Carlo baselines (Table 1), the kernel ablation (Table 2), and the bound-tightness plot (Figure 2).

`RUN_MODE = "smoke"` runs the whole thing at toy scale in a few minutes so you can check every cell executes. `RUN_MODE = "full"` uses the settings described in the paper where they are feasible on a single GPU, and states every place they are scaled down (the Monte Carlo baseline in particular; see the notes at the end). Results are written to `results/`.

In [ ]:
RUN_MODE = "full"          # "smoke" or "full"
SAVE_TO_DRIVE = False

import os, math, time, json, random, urllib.request
from dataclasses import dataclass
from typing import Callable, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.optimize as sopt
import scipy.stats as st
import pandas as pd
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float32)
print("device:", DEVICE, "| torch", torch.__version__)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/shapeuq_results"
else:
    OUT = "results"
os.makedirs(OUT, exist_ok=True)

if RUN_MODE == "smoke":
    CFG = dict(sdf_iters=200, pde_iters=200, ref_iters=300, mc_warm_iters=20, mc_samples=4,
               n_obs=1000, n_dom=1000, n_bnd=200, sdf_hidden=64, sdf_layers=4, pde_hidden=48, pde_layers=3,
               test_cases=1, mc_cases=1, noise_levels=[0.05], domains=["glacier", "protein", "coral"], gp_points=100)
else:
    CFG = dict(sdf_iters=3000, pde_iters=2500, ref_iters=8000, mc_warm_iters=200, mc_samples=64,
               n_obs=20000, n_dom=10000, n_bnd=2000, sdf_hidden=256, sdf_layers=8, pde_hidden=128, pde_layers=4,
               test_cases=6, mc_cases=2, noise_levels=[0.01, 0.05, 0.10], domains=["glacier", "protein", "coral"], gp_points=400)
print(json.dumps(CFG, indent=1))

def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## 1. Geometries: the three SciUQ-3D domains

Each domain is an exact signed distance function on the unit box, which plays the role of the ground-truth geometry phi-star. Measurements are surface points with Gaussian noise of sigma times the domain size, like the paper's corrupted altimetry, van der Waals and sonar clouds.

The glacier is a bed-to-surface slab under a rough heightfield (the ICESat-2 tracks the paper used need a NASA Earthdata login, so the surface here is a synthetic Jakobshavn-like profile). The protein surface is the union of van der Waals spheres of the real atoms of 1UBQ from the PDB (downloaded in the notebook; a synthetic blob is used if the download fails). The coral is a union of capsules forming a branching thin-plate structure.

In [ ]:
class Geometry:
    """An exact SDF phi(x) < 0 inside the body. Coordinates live in [-1, 1]^3; size = 2."""
    name = "base"
    size = 2.0

    def sdf(self, x: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError

    def normal(self, x: torch.Tensor) -> torch.Tensor:
        x = x.detach().requires_grad_(True)
        g, = torch.autograd.grad(self.sdf(x).sum(), x)
        return g / (g.norm(dim=1, keepdim=True) + 1e-9)

    def sample_surface(self, n: int, gen: torch.Generator, steps: int = 8) -> torch.Tensor:
        """Rejection-sample points near the surface, then project onto phi = 0 with Newton steps."""
        pts = []
        while sum(len(p) for p in pts) < n:
            x = (torch.rand(4 * n, 3, generator=gen) * 2 - 1).to(DEVICE)
            with torch.no_grad():
                d = self.sdf(x)
            pts.append(x[d.abs() < 0.15])
        x = torch.cat(pts)[:n]
        for _ in range(steps):
            x = x.detach().requires_grad_(True)
            d = self.sdf(x)
            g, = torch.autograd.grad(d.sum(), x)
            x = (x - (d / (g.norm(dim=1) ** 2 + 1e-9)).unsqueeze(1) * g).detach()
        return x.clamp(-0.999, 0.999)


class Glacier(Geometry):
    """Ice slab: bed at z = -0.6, rough surface z = h(x, y). Inside where bed < z < h."""
    name = "glacier"

    def __init__(self):
        g = torch.Generator().manual_seed(11)
        self.amp = torch.rand(6, generator=g) * 0.08 + 0.02
        self.fx = torch.rand(6, generator=g) * 3 + 1
        self.fy = torch.rand(6, generator=g) * 3 + 1
        self.ph = torch.rand(6, generator=g) * 6.28

    def height(self, xy):
        h = 0.25 - 0.35 * xy[:, 0]
        for a, fx, fy, p in zip(self.amp, self.fx, self.fy, self.ph):
            h = h + a * torch.sin(fx * math.pi * xy[:, 0] + fy * math.pi * xy[:, 1] + p)
        return h

    def sdf(self, x):
        top = x[:, 2] - self.height(x[:, :2])
        bed = -0.6 - x[:, 2]
        lateral = torch.stack([x[:, 0].abs() - 0.85, x[:, 1].abs() - 0.85], 1).max(1).values
        return torch.stack([top, bed, lateral], 1).max(1).values


class Protein(Geometry):
    """Union of van der Waals spheres from PDB atom coordinates, rescaled into the unit box."""
    name = "protein"
    RADII = {"C": 1.7, "N": 1.55, "O": 1.52, "S": 1.8, "H": 1.1}

    def __init__(self, pdb_id="1UBQ"):
        coords, radii = self._load(pdb_id)
        c = coords - coords.mean(0)
        scale = 0.75 / np.abs(c).max()
        self.centers = torch.tensor(c * scale, dtype=torch.float32, device=DEVICE)
        self.radii = torch.tensor(radii * scale * 1.4, dtype=torch.float32, device=DEVICE)  # probe-inflated vdW
        self.pdb_id = pdb_id

    def _load(self, pdb_id):
        try:
            url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
            txt = urllib.request.urlopen(url, timeout=20).read().decode()
            coords, radii = [], []
            for line in txt.splitlines():
                if line.startswith("ATOM"):
                    el = line[76:78].strip() or line[12:14].strip()[0]
                    if el == "H":
                        continue
                    coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
                    radii.append(self.RADII.get(el, 1.7))
            print(f"loaded {len(coords)} heavy atoms from PDB {pdb_id}")
            return np.array(coords), np.array(radii)
        except Exception as e:
            print("PDB download failed, using a synthetic blob:", e)
            g = np.random.default_rng(3)
            coords = g.normal(size=(300, 3)) * np.array([6, 4, 3])
            return coords, np.full(300, 1.7)

    def sdf(self, x):
        d = torch.cdist(x, self.centers) - self.radii[None, :]
        return (-torch.logsumexp(-d * 40.0, dim=1) / 40.0)


class Coral(Geometry):
    """Branching capsules (thin plates with curvature singularities), the hardest surface of the three."""
    name = "coral"

    def __init__(self):
        g = torch.Generator().manual_seed(7)
        segs = [(torch.tensor([0., 0., -0.9]), torch.tensor([0., 0., -0.3]), 0.12)]
        tips = [torch.tensor([0., 0., -0.3])]
        for depth in range(3):
            new = []
            for t in tips:
                for _ in range(2):
                    d = torch.randn(3, generator=g); d[2] = d[2].abs() + 0.8
                    d = d / d.norm() * (0.45 - 0.1 * depth)
                    end = (t + d).clamp(-0.9, 0.9)
                    segs.append((t, end, 0.09 - 0.02 * depth)); new.append(end)
            tips = new
        self.a = torch.stack([s[0] for s in segs]).to(DEVICE)
        self.b = torch.stack([s[1] for s in segs]).to(DEVICE)
        self.r = torch.tensor([s[2] for s in segs], device=DEVICE)

    def sdf(self, x):
        ab = self.b - self.a
        ap = x[:, None, :] - self.a[None]
        t = ((ap * ab[None]).sum(-1) / (ab * ab).sum(-1)[None]).clamp(0, 1)
        d = (ap - t[..., None] * ab[None]).norm(dim=-1) - self.r[None]
        return -torch.logsumexp(-d * 40.0, dim=1) / 40.0


def make_geometry(name: str) -> Geometry:
    return {"glacier": Glacier, "protein": Protein, "coral": Coral}[name]()

## 2. Neural SDF reconstruction from noisy measurements

Measurements: surface points of the exact geometry plus isotropic Gaussian noise of sigma times the domain size. 80% train the SDF, 20% are held out for the GP. The SDF is an MLP with positional encoding trained with the point loss, the Eikonal loss and an off-surface term that keeps the network from collapsing to zero, in the IGR/NeuS spirit.

In [ ]:
class PosEnc(nn.Module):
    def __init__(self, n_freq=10):
        super().__init__()
        self.register_buffer("freqs", (2.0 ** torch.arange(n_freq)) * math.pi)

    def forward(self, x):
        xb = x[..., None] * self.freqs
        return torch.cat([x, torch.sin(xb).flatten(1), torch.cos(xb).flatten(1)], 1)


class MLP(nn.Module):
    def __init__(self, d_in, d_out, hidden, layers, pe_freqs=0, act=nn.Softplus(beta=100)):
        super().__init__()
        self.pe = PosEnc(pe_freqs) if pe_freqs > 0 else nn.Identity()
        d = d_in * (1 + 2 * pe_freqs) if pe_freqs > 0 else d_in
        mods = []
        for _ in range(layers):
            mods += [nn.Linear(d, hidden), act]
            d = hidden
        mods.append(nn.Linear(d, d_out))
        self.net = nn.Sequential(*mods)

    def forward(self, x):
        return self.net(self.pe(x))


class SDFNet(nn.Module):
    """IGR/NeuS style SDF: positional encoding, softplus, geometric initialisation to a sphere so the
    sign is well defined from step 0 (the PE columns of the first layer start at zero, as in NeuS)."""

    def __init__(self, hidden, layers, pe_freqs=10, radius=0.6):
        super().__init__()
        self.pe = PosEnc(pe_freqs)
        d_in = 3 * (1 + 2 * pe_freqs)
        dims = [d_in] + [hidden] * layers + [1]
        self.layers = nn.ModuleList()
        for i in range(len(dims) - 1):
            lin = nn.Linear(dims[i], dims[i + 1])
            if i == len(dims) - 2:                       # last layer: unit-direction weights, bias -radius
                nn.init.normal_(lin.weight, mean=math.sqrt(math.pi) / math.sqrt(dims[i]), std=1e-4)
                nn.init.constant_(lin.bias, -radius)
            elif i == 0:                                 # first layer: xyz columns normal, PE columns zero
                nn.init.constant_(lin.bias, 0.0)
                nn.init.normal_(lin.weight, 0.0, math.sqrt(2) / math.sqrt(dims[i + 1]))
                with torch.no_grad():
                    lin.weight[:, 3:] = 0.0
            else:
                nn.init.constant_(lin.bias, 0.0)
                nn.init.normal_(lin.weight, 0.0, math.sqrt(2) / math.sqrt(dims[i + 1]))
            self.layers.append(lin)
        self.act = nn.Softplus(beta=100)

    def forward(self, x):
        h = self.pe(x)
        for i, lin in enumerate(self.layers):
            h = lin(h)
            if i < len(self.layers) - 1:
                h = self.act(h)
        return h


def measure(geom: Geometry, sigma: float, n: int, seed: int):
    gen = torch.Generator().manual_seed(seed)
    pts = geom.sample_surface(n, gen)
    noise = torch.randn(pts.shape, generator=gen).to(DEVICE) * sigma * geom.size
    obs = pts + noise
    perm = torch.randperm(n, generator=gen).to(DEVICE)
    k = int(0.8 * n)
    return obs[perm[:k]], obs[perm[k:]]


def fit_sdf(train_pts: torch.Tensor, iters: int, seed: int) -> nn.Module:
    seed_everything(seed)
    net = SDFNet(CFG["sdf_hidden"], CFG["sdf_layers"], pe_freqs=10).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, iters)
    n = len(train_pts)
    nb = min(4096 if RUN_MODE == "full" else 512, n)
    for it in range(iters):
        idx = torch.randint(0, n, (nb,), device=DEVICE)
        p = train_pts[idx]
        free = (torch.rand(len(p), 3, device=DEVICE) * 2 - 1)
        x = torch.cat([p, free]).requires_grad_(True)
        phi = net(x).squeeze(1)
        g, = torch.autograd.grad(phi.sum(), x, create_graph=True)
        loss_pt = phi[: len(p)].abs().mean()
        loss_eik = ((g.norm(dim=1) - 1) ** 2).mean()
        loss_off = torch.exp(-100 * phi[len(p):].abs()).mean()
        loss = loss_pt + 0.1 * loss_eik + 0.1 * loss_off
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return net


def sdf_fn(net):
    return lambda x: net(x).squeeze(1)

## 3. The GP model of reconstruction error (Eq. 1)

Residuals delta-phi at held-out measurements are modelled as a zero-mean GP with a Matérn-3/2 kernel. Hyperparameters (variance, lengthscale, nugget) are fitted by maximum marginal likelihood with L-BFGS on a subsample.

In [ ]:
def kernel_matrix(kind: str, X: torch.Tensor, Y: torch.Tensor, params: dict) -> torch.Tensor:
    r = torch.cdist(X, Y) / params["ell"]
    v = params["var"]
    if kind == "rbf":
        return v * torch.exp(-0.5 * r ** 2)
    if kind == "matern12":
        return v * torch.exp(-r)
    if kind == "matern32":
        s = math.sqrt(3) * r
        return v * (1 + s) * torch.exp(-s)
    if kind == "matern52":
        s = math.sqrt(5) * r
        return v * (1 + s + s ** 2 / 3) * torch.exp(-s)
    if kind == "sm":   # spectral mixture, Q components, weights/means/scales in params
        K = torch.zeros_like(r)
        for w, mu, sc in zip(params["w"], params["mu"], params["sc"]):
            d = torch.cdist(X, Y)
            K = K + w * torch.exp(-2 * math.pi ** 2 * d ** 2 * sc ** 2) * torch.cos(2 * math.pi * d * mu)
        return v * K
    raise ValueError(kind)


@dataclass
class GP:
    kind: str
    params: dict
    nugget: float
    X: torch.Tensor
    y: torch.Tensor

    def K(self, A, B):
        return kernel_matrix(self.kind, A, B, self.params)

    def trace_on(self, pts):
        return self.K(pts, pts).diagonal().sum().item()


def fit_gp(kind: str, X: torch.Tensor, y: torch.Tensor, seed: int = 0) -> GP:
    """Type-II ML for (log var, log ell, log nugget); the spectral mixture keeps Q=4 fixed random components."""
    n = min(CFG["gp_points"], len(X))
    idx = torch.randperm(len(X), generator=torch.Generator().manual_seed(seed))[:n].to(X.device)
    Xs, ys = X[idx].double(), y[idx].double()
    ys = ys - ys.mean()
    q_rng = np.random.default_rng(seed)
    extra = {"w": [0.25] * 4, "mu": list(q_rng.uniform(0.2, 3.0, 4)), "sc": list(q_rng.uniform(0.5, 2.0, 4))} if kind == "sm" else {}

    def nll(theta):
        var, ell, nug = np.exp(theta)
        params = {"var": float(var), "ell": float(ell), **extra}
        K = kernel_matrix(kind, Xs, Xs, {k: (torch.tensor(v, dtype=torch.float64) if isinstance(v, float) else v) for k, v in params.items()}).double()
        K = K + (nug + 1e-6) * torch.eye(n, dtype=torch.float64, device=X.device)
        try:
            L = torch.linalg.cholesky(K)
        except Exception:
            return 1e6
        alpha = torch.cholesky_solve(ys[:, None], L)
        return float(0.5 * (ys[:, None] * alpha).sum() + L.diagonal().log().sum() + 0.5 * n * math.log(2 * math.pi))

    y_var = float(ys.var())
    res = sopt.minimize(nll, x0=np.log([y_var + 1e-8, 0.3, y_var * 0.1 + 1e-8]), method="L-BFGS-B",
                        bounds=[(math.log(1e-10), math.log(10)), (math.log(0.02), math.log(3.0)), (math.log(1e-10), math.log(1))])
    var, ell, nug = np.exp(res.x)
    params = {"var": float(var), "ell": float(ell), **extra}
    return GP(kind, params, float(nug), X, y)

## 4. Meshfree collocation PDE solvers, forward and adjoint

Domain points are sampled in the box and kept where phi < -delta; boundary points are projected onto phi = 0 along the SDF gradient. The solution network is trained on the PDE residual at domain points and the Dirichlet data at boundary points.

Three PDEs, each with its adjoint:
glacier: steady heat diffusion, minus the Laplacian of T equals q, T = T_surface on the boundary; quantity of interest the domain-average temperature.
protein: linearised Poisson-Boltzmann, minus the Laplacian of phi plus kappa squared phi equals rho; quantity of interest the mean potential in a shell around the molecule (a domain integral, so Theorem 3 applies directly).
coral: Stokes flow in the box outside the coral, no-slip on the coral, uniform inflow on the box faces; quantity of interest the mean streamwise velocity in the wake region. The Hadamard formula for Stokes with Dirichlet data is the same structure as Eq. 7 summed over velocity components.

All three use Dirichlet conditions on the reconstructed surface so that Theorem 3 holds as stated (the paper's glacier uses a Robin surface condition; that is the one simplification here).

In [ ]:
@dataclass
class PDE:
    name: str
    n_out: int                  # scalar PDEs: 1; Stokes: 4 (u, v, w, p)
    operator: Callable          # (x, u) -> L u, shape (N, n_comp)
    bc: Callable                # x on the body surface -> Dirichlet values (N, n_comp)
    source: Callable            # x -> f (N, n_comp)
    qoi_weight: Callable        # (x, phi) -> w(x) >= 0, J = int w u_1 dx / int w dx
    box_bc: Optional[Callable]  # x on the box faces -> values, or None (no box boundary)
    inside_is_domain: bool      # glacier and protein: PDE inside the body; coral: outside

    @property
    def n_comp(self):
        return min(self.n_out, 3)


def laplacian(u, x):
    g, = torch.autograd.grad(u.sum(), x, create_graph=True)
    lap = 0
    for i in range(3):
        gi, = torch.autograd.grad(g[:, i].sum(), x, create_graph=True)
        lap = lap + gi[:, i]
    return lap, g


def make_pde(name: str) -> PDE:
    zeros = lambda k: (lambda x: torch.zeros(len(x), k, device=x.device))
    ones = lambda k: (lambda x: torch.ones(len(x), k, device=x.device))
    if name == "glacier":
        def op(x, u):
            lap, _ = laplacian(u[:, 0], x)
            return (-lap)[:, None]
        return PDE("glacier", 1, op, bc=lambda x: -0.5 + 0.2 * x[:, 2:3], source=ones(1),
                   qoi_weight=lambda x, phi: torch.ones(len(x), device=x.device), box_bc=None, inside_is_domain=True)
    if name == "protein":
        kappa2 = 4.0
        def op(x, u):
            lap, _ = laplacian(u[:, 0], x)
            return (-lap + kappa2 * u[:, 0])[:, None]
        return PDE("protein", 1, op, bc=ones(1), source=zeros(1),
                   qoi_weight=lambda x, phi: (phi(x) < 0.25).float(), box_bc=zeros(1), inside_is_domain=False)
    if name == "coral":
        mu = 1.0
        def op(x, u):
            gp, = torch.autograd.grad(u[:, 3].sum(), x, create_graph=True)
            r, div = [], 0
            for i in range(3):
                lap, g = laplacian(u[:, i], x)
                div = div + g[:, i]
                r.append(-mu * lap + gp[:, i])
            r.append(div)
            return torch.stack(r, 1)
        inflow = lambda x: torch.cat([torch.ones(len(x), 1, device=x.device), torch.zeros(len(x), 2, device=x.device)], 1)
        return PDE("coral", 4, op, bc=zeros(3), source=zeros(3),
                   qoi_weight=lambda x, phi: ((x[:, 0] > 0.2) & (x[:, 2] < 0.3)).float(), box_bc=inflow, inside_is_domain=False)
    raise ValueError(name)


def sample_domain(phi: Callable, inside: bool, n_dom: int, n_bnd: int, gen: torch.Generator, delta=0.01):
    """Interior collocation points and projected boundary points (Section 5, Step 1)."""
    dom, n_tried, n_kept, n_shell = [], 0, 0, 0
    eps = 0.02
    for _attempt in range(200):
        if sum(len(d) for d in dom) >= n_dom:
            break
        x = (torch.rand(4 * n_dom, 3, generator=gen) * 2 - 1).to(DEVICE)
        with torch.no_grad():
            d = phi(x)
        keep = (d < -delta) if inside else (d > delta)
        n_tried += len(x); n_kept += int(keep.sum()); n_shell += int((d.abs() < eps).sum())
        dom.append(x[keep])
    dom = torch.cat(dom)
    if len(dom) < n_dom // 2:
        raise RuntimeError("could not sample the domain: the SDF has no interior/exterior region of the requested sign")
    dom = dom[:n_dom]
    volume = 8.0 * n_kept / n_tried                 # |Omega| by Monte Carlo over the box
    area = 8.0 * n_shell / n_tried / (2 * eps)      # |Sigma| from the volume of the thin shell |phi| < eps
    near = []
    for _attempt in range(200):
        if sum(len(b) for b in near) >= n_bnd:
            break
        x = (torch.rand(6 * n_bnd, 3, generator=gen) * 2 - 1).to(DEVICE)
        with torch.no_grad():
            d = phi(x)
        near.append(x[d.abs() < 0.1])
    x = torch.cat(near)[:n_bnd]
    for _ in range(10):
        x = x.detach().requires_grad_(True)
        d = phi(x)
        g, = torch.autograd.grad(d.sum(), x)
        x = (x - (d / (g.norm(dim=1) ** 2 + 1e-9)).unsqueeze(1) * g).detach()
    bnd = x.clamp(-0.999, 0.999)
    x = bnd.detach().requires_grad_(True)
    g, = torch.autograd.grad(phi(x).sum(), x)
    gnorm = g.norm(dim=1)
    normal = g / (gnorm[:, None] + 1e-9)
    if not inside:
        normal = -normal        # outward from the fluid/solvent region points into the body
    faces = (torch.rand(n_bnd, 3, generator=gen) * 2 - 1).to(DEVICE)
    axis = torch.randint(0, 3, (n_bnd,), generator=gen).to(DEVICE)
    sign = (torch.randint(0, 2, (n_bnd,), generator=gen).to(DEVICE) * 2 - 1).float()
    faces[torch.arange(n_bnd), axis] = sign
    return {"dom": dom, "bnd": bnd, "normal": normal, "gnorm": gnorm.detach(), "box": faces, "volume": volume, "area": area}


def solve_pde(pde: PDE, phi: Callable, pts: dict, iters: int, seed: int, adjoint=False,
              init: Optional[nn.Module] = None, qoi_norm: float = 1.0):
    """Collocation solve. Forward: L u = f, u = bc on the surface (and box_bc on the box).
    Adjoint: L* p = j (j = w / qoi_norm on the first component), p = 0 on every boundary."""
    dom, bnd, box = pts["dom"], pts["bnd"], pts["box"]
    if init is None:
        seed_everything(seed)
        net = MLP(3, pde.n_out, CFG["pde_hidden"], CFG["pde_layers"], pe_freqs=4, act=nn.Tanh()).to(DEVICE)
    else:
        net = init
    opt = torch.optim.Adam(net.parameters(), lr=2e-3 if init is None else 5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, iters)
    nb = min(len(dom), 4096 if RUN_MODE == "full" else 512)
    nc = pde.n_comp
    for it in range(iters):
        idx = torch.randint(0, len(dom), (nb,), device=DEVICE)
        x = dom[idx].detach().requires_grad_(True)
        r = pde.operator(x, net(x))
        rhs = torch.zeros(nb, r.shape[1], device=DEVICE)
        if adjoint:
            rhs[:, 0] = pde.qoi_weight(x, phi) / qoi_norm
        else:
            rhs[:, :nc] = pde.source(x)
        loss = ((r - rhs) ** 2).mean()
        ub = net(bnd)[:, :nc]
        target = torch.zeros_like(ub) if adjoint else pde.bc(bnd)
        loss = loss + 50.0 * ((ub - target) ** 2).mean()
        if pde.box_bc is not None:
            ubox = net(box)[:, :nc]
            tb = torch.zeros_like(ubox) if adjoint else pde.box_bc(box)
            loss = loss + 50.0 * ((ubox - tb) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return net


def qoi(pde: PDE, net: nn.Module, pts: dict, phi: Callable) -> float:
    """J = int w u_1 dx / int w dx, by Monte Carlo over the collocation points."""
    with torch.no_grad():
        u = net(pts["dom"])[:, 0]
        w = pde.qoi_weight(pts["dom"], phi)
        return float((w * u).sum() / (w.sum() + 1e-9))


def qoi_norm(pde: PDE, pts: dict, phi: Callable) -> float:
    """int w dx, so that j = w / qoi_norm integrates the QoI exactly as qoi() computes it."""
    with torch.no_grad():
        w = pde.qoi_weight(pts["dom"], phi)
        return float(pts["volume"] * w.mean() + 1e-9)


def normal_derivative(net: nn.Module, bnd: torch.Tensor, normal: torch.Tensor, n_comp: int) -> torch.Tensor:
    x = bnd.detach().requires_grad_(True)
    u = net(x)
    out = []
    for i in range(n_comp):
        g, = torch.autograd.grad(u[:, i].sum(), x, retain_graph=True)
        out.append((g * normal).sum(1))
    return torch.stack(out, 1).detach()

## 5. ShapeUQ: sensitivity field, GP propagation, confidence interval

Theorem 3: with u the forward solution and p the adjoint, the sensitivity of the quantity of interest to a normal displacement at boundary point x is minus the product of the two normal derivatives, divided by the gradient norm of the SDF at that point (Eq. 7). With the GP covariance K evaluated at the boundary points, the variance of the QoI is S transposed K S (Eq. 8), and the 90% interval is J plus or minus 1.645 times its square root.

The first-order Perturbation baseline uses the same S with an independent-noise covariance (the fitted GP variance times the identity, no spatial correlation), which is what "operator norm bound without the GP" amounts to in practice.

In [ ]:
Z90 = st.norm.ppf(0.95)


def shapeuq(pde, u_net, p_net, pts: dict, gp: GP):
    bnd, normal, gnorm = pts["bnd"], pts["normal"], pts["gnorm"]
    du = normal_derivative(u_net, bnd, normal, pde.n_comp)
    dp = normal_derivative(p_net, bnd, normal, pde.n_comp)
    S = -(du * dp).sum(1) / (gnorm + 1e-6)              # Eq. 7 at each boundary point
    S = S * (pts["area"] / len(bnd))                     # surface quadrature weight
    K = gp.K(bnd, bnd)
    var = float(S @ K @ S)                               # Eq. 8
    var_pert = float((S ** 2).sum() * gp.params["var"]) # no spatial correlation
    return S.detach(), var, var_pert, float(K.diagonal().sum())


def run_case(domain: str, sigma: float, seed: int, kernels=("matern32",), do_mc=False, ref_cache=None, verbose=True):
    t_all = time.time()
    geom = make_geometry(domain) if ref_cache is None else ref_cache["geom"]
    pde = make_pde(domain)
    gen = torch.Generator().manual_seed(seed)

    # reference solution on the exact geometry (the paper used FEM; here the same collocation solver with more capacity/iterations)
    if ref_cache is None or "J_ref" not in ref_cache:
        pts_ref = sample_domain(geom.sdf, pde.inside_is_domain, CFG["n_dom"], CFG["n_bnd"], torch.Generator().manual_seed(999))
        u_ref = solve_pde(pde, geom.sdf, pts_ref, CFG["ref_iters"], 999)
        J_ref = qoi(pde, u_ref, pts_ref, geom.sdf)
        ref_cache = {"geom": geom, "J_ref": J_ref}
    J_ref = ref_cache["J_ref"]

    # measurements, reconstruction, GP
    train, held = measure(geom, sigma, CFG["n_obs"], seed)
    sdf_net = fit_sdf(train, CFG["sdf_iters"], seed)
    phi = sdf_fn(sdf_net)
    with torch.no_grad():
        resid = phi(held)                                    # phi_theta at points that should be on the surface
    t_gp = time.time()
    gps = {k: fit_gp(k, held, resid, seed) for k in kernels}
    gp = gps[kernels[0]]
    t_gp = (time.time() - t_gp) / len(kernels)

    # forward + adjoint on the reconstructed domain
    pts = sample_domain(phi, pde.inside_is_domain, CFG["n_dom"], CFG["n_bnd"], gen)
    t0 = time.time()
    u_net = solve_pde(pde, phi, pts, CFG["pde_iters"], seed)
    t_forward = time.time() - t0
    J = qoi(pde, u_net, pts, phi)
    p_net = solve_pde(pde, phi, pts, CFG["pde_iters"], seed + 1, adjoint=True, qoi_norm=qoi_norm(pde, pts, phi))
    S, var, var_pert, trK = shapeuq(pde, u_net, p_net, pts, gp)
    t_shapeuq = time.time() - t0 + t_gp                # forward + adjoint + GP fit
    kernel_results = {}
    for k, g_k in gps.items():
        _, v_k, _, _ = shapeuq(pde, u_net, p_net, pts, g_k)
        kernel_results[k] = {"half": Z90 * math.sqrt(max(v_k, 0)), "cover": abs(J - J_ref) <= Z90 * math.sqrt(max(v_k, 0))}

    out = {"domain": domain, "sigma": sigma, "seed": seed, "kernel": kernels[0], "J_ref": J_ref, "J": J,
           "t_forward": t_forward, "speedup_mc500_projected": 500 * t_forward / max(t_shapeuq, 1e-9),
           "kernel_results": kernel_results,
           "half_shapeuq": float(Z90 * math.sqrt(max(var, 0))), "half_pert": float(Z90 * math.sqrt(max(var_pert, 0))),
           "cover_shapeuq": bool(abs(J - J_ref) <= Z90 * math.sqrt(max(var, 0))),
           "cover_pert": bool(abs(J - J_ref) <= Z90 * math.sqrt(max(var_pert, 0))),
           "trK": trK, "var_shapeuq": var, "t_shapeuq": t_shapeuq, "gp_var": gp.params["var"], "gp_ell": gp.params["ell"]}

    # Monte Carlo over geometry samples drawn from the fitted GP, each a warm-started re-solve
    if do_mc:
        import copy
        t0 = time.time()
        bnd, normal, gnorm = pts["bnd"], pts["normal"], pts["gnorm"]
        grad_dir = normal if pde.inside_is_domain else -normal      # direction of grad phi
        K = gp.K(bnd, bnd) + 1e-6 * torch.eye(len(bnd), device=DEVICE)
        L = torch.linalg.cholesky(K.double()).float()
        Js = []
        for m in range(CFG["mc_samples"]):
            dphi = L @ torch.randn(len(bnd), generator=gen).to(DEVICE)
            psi = dphi / (gnorm + 1e-6)
            pts_m = dict(pts); pts_m["bnd"] = bnd - psi[:, None] * grad_dir   # phi + dphi = 0 moves the surface by -dphi/|grad phi|
            net_m = solve_pde(pde, phi, pts_m, CFG["mc_warm_iters"], seed, init=copy.deepcopy(u_net))
            Js.append(qoi(pde, net_m, pts, phi))
        lo, hi = np.percentile(Js, [5, 95])
        out.update({"mc_lo": float(lo), "mc_hi": float(hi), "cover_mc": bool(lo <= J_ref <= hi), "mc_var": float(np.var(Js)),
                    "t_mc": time.time() - t0, "speedup": (time.time() - t0) / max(t_shapeuq, 1e-9)})
    out["t_total"] = time.time() - t_all
    if verbose:
        print(f"{domain:8s} sigma={sigma:.2f} seed={seed} J_ref={J_ref:.4f} J={J:.4f} ±{out['half_shapeuq']:.4f} "
              f"cover={out['cover_shapeuq']} pert_cover={out['cover_pert']}" + (f" mc_cover={out['cover_mc']} speedup={out['speedup']:.1f}x" if do_mc else "")
              + f" ({out['t_total']:.0f}s)")
    return out, ref_cache

Quick single case to check the pipeline end to end.

In [ ]:
o, cache = run_case("glacier", 0.05, 0, do_mc=(RUN_MODE == "smoke"))
print({k: v for k, v in o.items() if k != "kernel_results"})

## 6. Table 1: coverage and speedup on SciUQ-3D

In [ ]:
ckpt = f"{OUT}/raw_cases.csv"
rows = pd.read_csv(ckpt).to_dict("records") if os.path.exists(ckpt) else []   # resume after a runtime restart
done = {(r["domain"], round(r["sigma"], 3), r["seed"]) for r in rows}
caches = {}
for domain in CFG["domains"]:
    for sigma in CFG["noise_levels"]:
        for case in range(CFG["test_cases"]):
            seed = 100 * case + int(sigma * 100)
            if (domain, round(sigma, 3), seed) in done:
                continue
            o, caches[domain] = run_case(domain, sigma, seed, do_mc=(case < CFG["mc_cases"]), ref_cache=caches.get(domain))
            o = {k: v for k, v in o.items() if k != "kernel_results"}
            rows.append(o)
            pd.DataFrame(rows).to_csv(ckpt, index=False)
df = pd.DataFrame(rows)

def table1(df):
    recs = []
    for domain in CFG["domains"]:
        for method in ["Deterministic", "Pert. (1st)", f"MC-{CFG['mc_samples']}", "ShapeUQ"]:
            r = {"Domain": domain, "Method": method}
            for sigma in CFG["noise_levels"]:
                sub = df[(df.domain == domain) & (df.sigma == sigma)]
                col = f"sigma={int(sigma*100)}%"
                if method == "Deterministic":
                    r[col] = "—"; r["Speedup"] = "1x"
                elif method.startswith("Pert"):
                    r[col] = f"{sub.cover_pert.mean():.2f}"; r["Speedup"] = "1x"
                elif method.startswith("MC"):
                    mc = sub.dropna(subset=["cover_mc"]) if "cover_mc" in sub else sub.iloc[0:0]
                    r[col] = f"{mc.cover_mc.mean():.2f}" if len(mc) else "n/a"; r["Speedup"] = "1x"
                else:
                    r[col] = f"{sub.cover_shapeuq.mean():.2f}"
                    mc = df[(df.domain == domain)].dropna(subset=["speedup"]) if "speedup" in df else df.iloc[0:0]
                    r["Speedup"] = (f"{mc.speedup.mean():.1f}x vs MC-{CFG['mc_samples']} warm-started; "
                                    f"{sub.speedup_mc500_projected.mean():.0f}x vs MC-500 full solves (projected)") if len(mc) else "n/a"
            recs.append(r)
    return pd.DataFrame(recs)

t1 = table1(df)
t1.to_csv(f"{OUT}/table1_coverage.csv", index=False)
print("Table 1: coverage = fraction of test cases where the reference QoI lies inside the 90% CI")
print(t1.to_string(index=False))

## 7. Table 2: GP kernel ablation on the glacier at sigma = 5%

In [ ]:
KERNELS = ["rbf", "matern12", "matern32", "matern52", "sm"]
abl = []
for case in range(max(2, CFG["test_cases"])):
    o, caches["glacier"] = run_case("glacier", 0.05, 500 + case, kernels=tuple(KERNELS), ref_cache=caches.get("glacier"), verbose=False)
    for k, kr in o["kernel_results"].items():
        abl.append({"kernel": k, "seed": o["seed"], "cover": kr["cover"], "half": kr["half"]})
abl = pd.DataFrame(abl)
t2 = abl.groupby("kernel", sort=False).agg(Coverage=("cover", "mean"), CI_width=("half", lambda s: 2 * s.mean())).round(4)
t2.to_csv(f"{OUT}/table2_kernels.csv")
print("Table 2: kernel ablation, glacier, sigma = 5%")
print(t2.to_string())

## 8. Figure 2: the Theorem 2 bound against the Monte Carlo variance

The bound says the expected squared error scales with the trace of K times the squared sup-norm of the normal derivative of the reference solution over the squared minimum gradient. We plot the ShapeUQ variance (the first-order term) and the MC variance against tr(K) across the cases where MC ran, and report the bound-to-empirical ratio.

In [ ]:
mc = df.dropna(subset=["mc_var"]) if "mc_var" in df else df.iloc[0:0]
if len(mc):
    fig, ax = plt.subplots(figsize=(5, 4))
    for domain, g in mc.groupby("domain"):
        ax.scatter(g.trK, g.mc_var, label=f"{domain} MC", marker="o")
        ax.scatter(g.trK, g.var_shapeuq, label=f"{domain} ShapeUQ", marker="x")
    ax.set_xlabel("tr(K)"); ax.set_ylabel("Var[J]"); ax.set_xscale("log"); ax.set_yscale("log"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f"{OUT}/figure2_bound.png", dpi=150); plt.show()
    ratio = (mc.var_shapeuq / mc.mc_var.clip(lower=1e-12))
    print("ShapeUQ variance / MC variance: median %.2f, max %.2f" % (ratio.median(), ratio.max()))

## 9. Findings from this run and what differs from the paper

The numbers above are what this code produced. Places where this reproduction departs from the paper, all deliberate:

The paper's reference solutions came from FEniCS on a fine mesh; here the reference is the same collocation solver on the exact SDF with more iterations, so "coverage" measures agreement between two neural solves. The glacier surface is synthetic (the ICESat-2 tracks need an Earthdata login) and the glacier uses a Dirichlet surface temperature rather than the paper's Robin condition so Theorem 3 applies unchanged. The Monte Carlo baseline uses warm-started re-solves and fewer samples than 500 (see `CFG["mc_samples"]`), which makes it far cheaper than the paper's MC-500 of full solves, so the measured speedup is small; the table also reports the projected cost of MC-500 with full forward solves (500 times the measured forward-solve time over the ShapeUQ time), which is the comparison the paper made and the one that gives numbers in the tens. Coverage is computed over `CFG["test_cases"]` noise realisations per cell.

In [ ]:
print(df.groupby(["domain", "sigma"])[["cover_shapeuq", "cover_pert", "half_shapeuq", "half_pert", "t_shapeuq"]].mean().round(4).to_string())
print("\nAll CSVs and figures are in", os.path.abspath(OUT))